# 📊 Evaluating LLM Applications: From Vibes to Metrics

### Dinesh AI Academy | Day 5 — Memory, Guardrails & Evaluation

**Learning objective:**
By the end of this notebook you will be able to explain why "it looked right
when I tried it" doesn't scale, and build three concrete evaluation
techniques: reference-based scoring, LLM-as-a-judge, and agent-level task
evaluation.

**Where we left off:** we gave an assistant memory, then locked it down with
guardrails. The last open question: **how do you know any of it actually
works** — not just today, but after you tweak the prompt, swap the model, or
add a new tool three months from now?

## 1. Why "Vibes-Based" Testing Doesn't Scale

> **Trying a few prompts by hand and reading the replies is real testing —
> for exactly one version, on exactly one day.** The moment you change a
> system instruction, upgrade the model, or add a guardrail, you have no way
> to know what you *broke* unless you re-check, systematically, every case
> that used to work.

```text
"Vibes" testing:                    Systematic evaluation:

  Try prompt A -> looks good           Fixed test set (prompts + expected
  Try prompt B -> looks good           behavior) -> run through the app ->
  Change the prompt...                 score every case automatically ->
  Ship it. 🤞                          compare score to last time.

  Did A and B still work? 🤷          Any regression shows up immediately.
```

This is the same idea as a test suite in normal software engineering —
except the "assertions" are fuzzier, because there's rarely one single
correct string of text. That fuzziness is exactly what the next three
sections address.

## 2. Setup — Gemini API Key

Same pattern as every earlier day: works locally (via `.env`) or in Google
Colab (via Secrets), without changing any code.

**Never publish your API key in a notebook, GitHub repository, Moodle,
WhatsApp group, or screenshot.**

In [2]:
# In Google Colab, run this cell once.
%pip -q install -U google-genai numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [3]:
from google import genai
from google.genai import types
import os

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

GAISTUDIO_API_KEY = get_secret("GAISTUDIO_API_KEY")

if not GAISTUDIO_API_KEY:
    raise ValueError(
        "GAISTUDIO_API_KEY not found. Set it in Colab Secrets or in your local .env file."
    )

client = genai.Client(api_key=GAISTUDIO_API_KEY)
CHAT_MODEL = "gemini-3.5-flash-lite"
EMBED_MODEL = "gemini-embedding-001"

print("Gemini client is ready.")
print("Chat model:", CHAT_MODEL, "| Embedding model:", EMBED_MODEL)

Gemini client is ready.
Chat model: gemini-3.5-flash-lite | Embedding model: gemini-embedding-001


## 3. Three Ways to Score an LLM's Answer

| Approach | How it works | Strength | Weakness |
|---|---|---|---|
| **Reference-based** | Compare the model's answer to a known-correct "golden" answer (exact match, or semantic similarity) | Fast, cheap, fully repeatable | Penalizes correct answers phrased differently |
| **LLM-as-a-judge** | Ask a second LLM call to grade the answer against a rubric | Handles paraphrasing, nuance, partial credit | Costs another API call; can inherit the judge model's own biases |
| **Human evaluation** | A person reads the answer and rates it | Catches everything the other two miss | Slow, expensive, doesn't scale to every prompt/model change |

Real pipelines use reference-based checks for anything with a clear right
answer, LLM-as-a-judge for anything more open-ended, and a *small* rotating
sample of human review to sanity-check that the judge itself is trustworthy.

## 4. Build a Tiny Evaluation Dataset

An eval set is just a list of `{prompt, expected}` pairs — the same idea as
test cases in normal software testing, except `expected` here is "the
substance of a correct answer," not a byte-for-byte string.

In [4]:
eval_dataset = [
    {"prompt": "What is the capital of France?", "expected": "Paris"},
    {"prompt": "What is 12 multiplied by 8?", "expected": "96"},
    {"prompt": "Who wrote the play Romeo and Juliet?", "expected": "William Shakespeare"},
    {"prompt": "What is the boiling point of water in Celsius, at sea level?", "expected": "100 degrees Celsius"},
    {"prompt": "Name the largest planet in our solar system.", "expected": "Jupiter"},
]

def run_model(prompt: str) -> str:
    return client.models.generate_content(model=CHAT_MODEL, contents=prompt).text.strip()

predictions = [{"prompt": item["prompt"], "expected": item["expected"], "actual": run_model(item["prompt"])}
               for item in eval_dataset]

for p in predictions:
    print(f"Q: {p['prompt']}\n  expected: {p['expected']}\n  actual  : {p['actual']}\n")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Q: What is the capital of France?
  expected: Paris
  actual  : The capital of France is Paris.

Q: What is 12 multiplied by 8?
  expected: 96
  actual  : 12 multiplied by 8 is 96.

Q: Who wrote the play Romeo and Juliet?
  expected: William Shakespeare
  actual  : The play *Romeo and Juliet* was written by **William Shakespeare**.

Q: What is the boiling point of water in Celsius, at sea level?
  expected: 100 degrees Celsius
  actual  : The boiling point of water in Celsius, at sea level, is **100°C**.

Q: Name the largest planet in our solar system.
  expected: Jupiter
  actual  : The largest planet in our solar system is **Jupiter**.



## 5. Reference-Based Scoring

### 5a. Exact / substring match — the cheapest possible check

Works well when the expected answer is a short fact that should appear
verbatim somewhere in the response.

In [5]:
def substring_match(expected: str, actual: str) -> bool:
    return expected.lower() in actual.lower()

for p in predictions:
    p["substring_score"] = substring_match(p["expected"], p["actual"])
    print(f"{p['expected']!r:35} in response? {p['substring_score']}")

'Paris'                             in response? True
'96'                                in response? True
'William Shakespeare'               in response? True
'100 degrees Celsius'               in response? False
'Jupiter'                           in response? True


### 5b. Semantic similarity — for answers that won't match word-for-word

A model might correctly answer "It boils at 100°C" instead of "100 degrees
Celsius" — substring match fails that, even though it's right. Embedding both
the expected and actual answers and comparing with cosine similarity (same
technique as Day 2's RAG notebook, and this day's Memory notebook) is more
forgiving of *phrasing*, while still penalizing answers that are actually
wrong.

In [6]:
import numpy as np

def embed(text: str) -> np.ndarray:
    result = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return np.array(result.embeddings[0].values)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

for p in predictions:
    p["semantic_score"] = cosine_similarity(embed(p["expected"]), embed(p["actual"]))
    print(f"{p['prompt'][:45]:45} similarity={p['semantic_score']:.3f}  substring={p['substring_score']}")

What is the capital of France?                similarity=0.725  substring=True
What is 12 multiplied by 8?                   similarity=0.668  substring=True
Who wrote the play Romeo and Juliet?          similarity=0.721  substring=True
What is the boiling point of water in Celsius similarity=0.785  substring=False
Name the largest planet in our solar system.  similarity=0.722  substring=True


## 6. LLM-as-a-Judge — Scoring What Similarity Can't

Reference-based scores struggle with anything that isn't a short fact —
helpfulness, tone, whether an explanation is actually correct even if none of
the exact reference words appear. The fix: ask a second Gemini call to grade
the answer against a rubric, and force it to return a strict, structured
score so you can aggregate it like any other number.

In [7]:
judge_schema = {
    "type": "object",
    "properties": {
        "correct": {"type": "boolean"},
        "score_1_to_5": {"type": "integer"},
        "reasoning": {"type": "string"},
    },
    "required": ["correct", "score_1_to_5", "reasoning"],
}

def llm_judge(prompt: str, expected: str, actual: str) -> dict:
    judge_prompt = (
        "You are grading an AI assistant's answer against a reference answer.\n"
        f"Question: {prompt}\n"
        f"Reference (correct) answer: {expected}\n"
        f"Assistant's actual answer: {actual}\n\n"
        "Judge whether the assistant's answer is factually correct, even if "
        "worded differently from the reference. Give a score from 1 (completely "
        "wrong) to 5 (fully correct and well stated)."
    )
    response = client.models.generate_content(
        model=CHAT_MODEL,
        contents=judge_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=judge_schema,
        ),
    )
    import json
    return json.loads(response.text)

for p in predictions:
    verdict = llm_judge(p["prompt"], p["expected"], p["actual"])
    p["judge_score"] = verdict["score_1_to_5"]
    print(f"{p['prompt'][:45]:45} judge={verdict['score_1_to_5']}/5  correct={verdict['correct']}  — {verdict['reasoning']}")

What is the capital of France?                judge=5/5  correct=True  — The assistant's answer correctly identifies Paris as the capital of France and is well stated.
What is 12 multiplied by 8?                   judge=5/5  correct=True  — The assistant correctly answered the mathematical question with the right product, 96, and formulated it in a clear, complete sentence.
Who wrote the play Romeo and Juliet?          judge=5/5  correct=True  — The assistant correctly identified William Shakespeare as the author of Romeo and Juliet.
What is the boiling point of water in Celsius judge=5/5  correct=True  — The assistant correctly identified the boiling point of water as 100°C, matching the reference answer.
Name the largest planet in our solar system.  judge=5/5  correct=True  — The assistant correctly identified Jupiter as the largest planet in our solar system, matching the reference answer.


Compare the three columns across your results: `substring_score`,
`semantic_score`, and `judge_score`. Whenever the model rephrases a correct
answer, expect `substring_score` to be the most punishing of the three, and
`judge_score` to be the most forgiving — that gap *is* the reason both
techniques coexist in real eval suites rather than picking just one.

## 7. Evaluating an Agent, Not Just a Single Answer

Everything so far scores a single question-answer pair. An **agent** (Day 4)
makes a *sequence* of decisions, so a good agent eval also checks: did it
reach the goal, how many steps did it take, and did it call the right tools?
Here's a minimal agent — the same shape as Day 4's `run_agent()` — wired up
with a tiny eval harness around it.

In [ ]:
def calculate(a: float, b: float, operation: str) -> float:
    ops = {"add": a + b, "subtract": a - b, "multiply": a * b, "divide": a / b if b else None}
    return ops[operation]

TOOLBOX = {"calculate": calculate}

calculator_declaration = {
    "name": "calculate",
    "description": "Performs basic arithmetic calculations.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number"},
            "b": {"type": "number"},
            "operation": {"type": "string", "enum": ["add", "subtract", "multiply", "divide"]},
        },
        "required": ["a", "b", "operation"],
    },
}

agent_config = types.GenerateContentConfig(
    tools=[types.Tool(function_declarations=[calculator_declaration])],
)

def run_agent(goal: str, max_steps: int = 4) -> dict:
    """A minimal agent loop (see Day 4) that also RETURNS eval-relevant metadata."""
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=goal)])]
    tool_calls_made = []

    for _ in range(max_steps):
        response = client.models.generate_content(model=CHAT_MODEL, contents=contents, config=agent_config)
        model_turn = response.candidates[0].content
        contents.append(model_turn)

        # Here we are only extracting the tool's functions from the model's response, and calling them in Python. 
        # The model itself is not executing any code.
        function_calls = [p.function_call for p in model_turn.parts if p.function_call]
        if not function_calls:
            # If the model didn't call any tools, we can return the answer and the steps taken.
            return {"answer": response.text, "steps": len(tool_calls_made), "tool_calls": tool_calls_made}

        # Now we will call the functions in Python and append the results to the conversation.
        result_parts = []
        for fc in function_calls:
            # Here we reading the arguments from the model's function.
            args = dict(fc.args)
            # Here we are calling the function in Python, and storing the result.
            result = TOOLBOX[fc.name](**args) # This is where we are calling the function 
            tool_calls_made.append({"name": fc.name, "args": args, "result": result})
            # Here we are appending the result of the function call to the conversation, so that the model can see it in the next turn.
            result_parts.append(types.Part.from_function_response(name=fc.name, response={"result": result}))
        contents.append(types.Content(role="user", parts=result_parts))

    return {"answer": "(hit step limit)", "steps": len(tool_calls_made), "tool_calls": tool_calls_made}


agent_eval_set = [
    {"goal": "What is 45 multiplied by 12?", "expected_tool": "calculate", "expected_answer": "540"},
    {"goal": "In one sentence, what is an AI agent?", "expected_tool": None, "expected_answer": None},
]

for case in agent_eval_set:
    result = run_agent(case["goal"])
    used_expected_tool = (
        case["expected_tool"] is None and result["steps"] == 0
    ) or any(tc["name"] == case["expected_tool"] for tc in result["tool_calls"])
    print(f"GOAL: {case['goal']}")
    print(f"  answer: {result['answer']}")
    print(f"  steps taken: {result['steps']}  |  used expected tool correctly: {used_expected_tool}")
    print()

Notice the extra dimensions beyond "was the final text correct": **did it use
a tool when it should have (and skip one when it shouldn't)**, and **how many
steps did it take to get there**. An agent that reaches the right answer in 6
steps when 1 would do is technically "correct" but inefficient — a good agent
eval catches that too, not just the final answer.

## 8. Aggregating Results — Pass Rates, Not Just Individual Scores

A single score tells you about one prompt. A **pass rate** across the whole
eval set tells you whether the *system* is healthy — and that's the number
you track over time as prompts and models change.

In [ ]:
PASS_THRESHOLD = 4  # judge score out of 5

pass_count = sum(1 for p in predictions if p["judge_score"] >= PASS_THRESHOLD)
total = len(predictions)
print(f"Pass rate: {pass_count}/{total} ({100 * pass_count / total:.0f}%)")

import matplotlib.pyplot as plt

labels = [p["prompt"][:20] + "..." for p in predictions]
scores = [p["judge_score"] for p in predictions]
colors = ["#2e7d32" if s >= PASS_THRESHOLD else "#c62828" for s in scores]

plt.figure(figsize=(8, 4))
plt.bar(labels, scores, color=colors)
plt.axhline(PASS_THRESHOLD, color="gray", linestyle="--", label=f"pass threshold ({PASS_THRESHOLD})")
plt.ylabel("Judge score (1-5)")
plt.xticks(rotation=30, ha="right")
plt.legend()
plt.title("Eval results by prompt")
plt.tight_layout()
plt.show()

## 9. Regression Testing — Turning This Into a Repeatable Check

The real value of an eval set isn't running it once — it's running the exact
same set again every time something changes, and immediately seeing if the
pass rate dropped. Wrap it in one function that reports a clear verdict.

In [ ]:
def run_eval_suite(dataset, pass_threshold: int = PASS_THRESHOLD) -> dict:
    results = []
    for item in dataset:
        actual = run_model(item["prompt"])
        verdict = llm_judge(item["prompt"], item["expected"], actual)
        results.append({**item, "actual": actual, "judge_score": verdict["score_1_to_5"]})

    passed = sum(1 for r in results if r["judge_score"] >= pass_threshold)
    pass_rate = passed / len(results)
    print(f"Eval suite: {passed}/{len(results)} passed ({pass_rate:.0%})")
    for r in results:
        status = "PASS" if r["judge_score"] >= pass_threshold else "FAIL"
        print(f"  [{status}] score={r['judge_score']}  {r['prompt'][:50]}")
    return {"pass_rate": pass_rate, "results": results}

# Run once now. Re-run this exact cell any time you change a prompt, a model,
# or a guardrail -- a dropping pass rate is your regression signal.
suite_result = run_eval_suite(eval_dataset)

## 🛡️ 10. Production Note — Where Evaluation Goes Wrong

- **Judge bias** — using the *same* model to both answer and judge tends to
  rate its own style favorably. Where possible, judge with a different model
  (or at least a fixed, versioned judge prompt you don't casually tweak).
- **Eval set leakage** — if your eval prompts end up in a future fine-tuning
  or few-shot prompt, you're no longer measuring generalization, just
  memorization of the test itself.
- **Overfitting to the eval set** — tuning prompts until every case in a
  10-example set passes doesn't guarantee the 11th real user question will.
  Grow the eval set as you discover new failure modes in production.
- **Cost at scale** — an LLM-as-judge call costs roughly as much as the
  original call. Running a large eval set on every commit adds up; budget
  for it like you would CI compute time.
- **Human spot checks stay necessary** — periodically have a person review a
  small random sample end-to-end, specifically to check whether the judge
  itself is still trustworthy.

## 🧪 11. Classroom Challenge

Add two new cases to `eval_dataset` — one the current prompt should clearly
pass, and one *tricky* case designed to expose a weakness (e.g. an answer
that's correct but phrased very differently from the reference, or a
borderline factual question). Re-run `run_eval_suite()` and check:

| Question | Substring match result | Judge result | Do they agree? |
|---|---|---|---|
| Your easy case | ? | ? | ? |
| Your tricky case | ? | ? | ? |

If they disagree, which one do you trust more for *this specific* case, and
why?

## 🎓 Day 5.3 Takeaway

By the end of this notebook, you should be able to explain:

1. Why manually trying a few prompts doesn't catch regressions once a
   prompt, model, or guardrail changes.
2. The three ways to score an LLM's output — reference-based, LLM-as-a-
   judge, human review — and when each one is the right tool.
3. Why agent evaluation needs more than "was the final answer right":
   task completion, step efficiency, and tool-call correctness.
4. Why a single **pass rate**, tracked over time, is more useful than any
   individual score — and the real pitfalls (judge bias, eval-set leakage,
   cost) that come with relying on it.

### The complete mental model

```text
   Eval dataset (prompt + expected)
              |
      Run through the app
              |
   +----------+-----------+
   |                       |
Reference-based        LLM-as-a-judge
(substring / cosine)    (rubric, structured score)
   |                       |
   +----------+-----------+
              |
        Pass / fail per case
              |
      Aggregate -> pass rate
              |
   Re-run on every prompt/model change
   -> pass rate drop = regression caught
```

## Official references

- Gemini API — Structured output: https://ai.google.dev/gemini-api/docs/structured-output
- Gemini API — Embeddings: https://ai.google.dev/gemini-api/docs/embeddings
- Stanford HELM (Holistic Evaluation of Language Models): https://crfm.stanford.edu/helm/
- OpenAI Evals framework: https://github.com/openai/evals
- promptfoo (open-source LLM eval/regression testing): https://www.promptfoo.dev/

This notebook built every scoring method from plain Python and direct Gemini
calls so the mechanics stay visible. Tools like promptfoo, DeepEval, and
OpenAI Evals automate exactly this loop — fixed dataset, multiple scoring
strategies, tracked pass rate over time — at much larger scale.